# v1 vs v2 BNN — predictive comparison on toy data

Visual sanity check that the v1 (`layer_sizes` + `unflatten_params`) and v2 (`ParamSpec` + `nn.Module` + `functional_call`) pipelines target the same posterior, even though `find_reference` and the sampler RNG paths produce different specific skeletons.

Both targets are mathematically identical — the bit-equivalence test (`module_bnn.py --check static`) confirms that. What we want to see here is that the predictive distributions over `x ∈ [-1, 1]` overlap.

## What we plot

1. **Predictive mean ± 2σ** for v1 and v2 overlaid on the same axes. If they agree, the bands should sit on top of each other.
2. A few **posterior sample functions** from each — visually checks that we're sampling from the same distribution of curves, not just matching first/second moments.
3. **Histogram of test-point predictions** for a single x, comparing the two posterior predictives directly. 

The expected result: bands and histograms overlap closely. They will *not* be identical because:
- `find_reference_bnn` runs Adam with different RNG-driven floating-point trajectories in the two pipelines
- The sampler is itself randomized
- 300 skeleton points is a short chain

Differences at the ~10% level on the predictive band are normal. Order-of-magnitude differences would indicate a bug.

## Setup

In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64
from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
# ===========================================================================
# ADJUST IMPORTS to match your project layout if these don't match.
# Same paths as the ones in module_bnn.py.
# ===========================================================================

# v1
from sazz.models.bnn_torch import (
    make_bnn_regression as v1_make_bnn_regression,
    predict_regression as v1_predict_regression,
)

# v2
from sazz.utils.bnn_modular_utils import (
    ParamSpec, build_prior_precision, build_ffn_module,
)

# Samplers + path resampling
from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.utils.sampling import resample_boomerang_path

# Model glue (used to build the v2 target)
from sazz.models.smoke_test import TorchTarget
from sazz.models.priors_torch import Prior
from sazz.models.models_torch import BayesianModel
from sazz.utils.warmup import find_reference_bnn

# Toy data loader
from sazz.scripts.uci_bnn import load_toy

## Local v2 likelihood

This is the same `_V2Likelihood` class used in `module_bnn.py`. Once you've added `ModuleLikelihood` to `bnn_torch.py`, replace this with the import.

In [ ]:
from torch.distributions import Normal

class V2Likelihood(nn.Module):
    """Functional-call-based BNN likelihood. Forward via torch.func.functional_call."""
    def __init__(self, module, spec, X, y, noise_std):
        super().__init__()
        self.module = module
        self.module.eval()
        self.spec = spec
        self.register_buffer('X', X)
        self.register_buffer('y', y)
        self.noise_std = noise_std

    def predict(self, beta, X_new):
        params = self.spec.to_dict(beta)
        return torch.func.functional_call(self.module, params, (X_new,))

    def log_prob_single(self, beta, X_i, y_i):
        preds = self.predict(beta, X_i).squeeze(-1)
        return Normal(preds, self.noise_std).log_prob(y_i).sum()

    def log_prob(self, beta):
        return self.log_prob_single(beta, self.X, self.y)


class V2Prior(Prior):
    def __init__(self, prec_vec):
        super().__init__()
        self.register_buffer('_precision', prec_vec)
    def log_prob(self, beta):
        return -0.5 * (self._precision * beta ** 2).sum()
    def precision_diag(self):
        return self._precision

## Load the toy and inspect

In [ ]:
DATASET = 'hernandez'   # try also 'gap', 'sharp', 'multiscale'

data, cfg = load_toy(DATASET)
X_train = data['X_train'].to(dtype=DTYPE)
y_train = data['y_train'].to(dtype=DTYPE)
X_test  = data['X_test'].to(dtype=DTYPE)
y_test  = data['y_test'].to(dtype=DTYPE)

print(f'dataset: {DATASET}')
print(f'  X_train: {tuple(X_train.shape)}, y_train: {tuple(y_train.shape)}')
print(f'  X_test:  {tuple(X_test.shape)},  y_test:  {tuple(y_test.shape)}')
print(f'  layer_sizes: {cfg.layer_sizes}, activation: {cfg.activation}')
print(f'  noise_std: {cfg.noise_std}')
print(f'  prior_std_weight: {cfg.prior_std_weight}, prior_std_bias: {cfg.prior_std_bias}')
print(f'  fan_in_scaling: {cfg.fan_in_scaling}')
print(f'  adam_steps: {cfg.adam_steps}')

## Build both targets

In [ ]:
# ---- v1 target ----
torch.manual_seed(0); np.random.seed(0)
t0 = time.perf_counter()
target_v1 = v1_make_bnn_regression(
    X_train, y_train,
    layer_sizes=cfg.layer_sizes,
    activation=cfg.activation,
    prior_std_weight=cfg.prior_std_weight,
    prior_std_bias=cfg.prior_std_bias,
    fan_in_scaling=cfg.fan_in_scaling,
    noise_std=cfg.noise_std,
    covariance_reference='laplace_diag',
    adam_steps=cfg.adam_steps,
    dtype=DTYPE,
)
print(f'v1 target built in {time.perf_counter() - t0:.2f}s  D={target_v1.D}')

# ---- v2 target ----
torch.manual_seed(0); np.random.seed(0)
t0 = time.perf_counter()
module_v2 = build_ffn_module(cfg.layer_sizes, cfg.activation).to(dtype=DTYPE)
spec_v2 = ParamSpec.from_module(module_v2)
prec_v2 = build_prior_precision(
    spec_v2, cfg.prior_std_weight, cfg.prior_std_bias,
    cfg.fan_in_scaling, DTYPE, 'cpu',
)
prior_v2 = V2Prior(prec_v2)
lik_v2 = V2Likelihood(module_v2, spec_v2, X_train, y_train, noise_std=cfg.noise_std)
model_v2 = BayesianModel(prior_v2, lik_v2)
x_ref_v2, Sigma_inv_v2 = find_reference_bnn(
    model_v2.energy, spec_v2.D, model=model_v2, dtype=DTYPE, device='cpu',
    reference='laplace_diag', n_steps=cfg.adam_steps, lr=1e-2,
)
target_v2 = TorchTarget(
    name='v2_bnn_regression',
    D=spec_v2.D,
    grad_target=model_v2.grad_energy,
    x_ref=x_ref_v2,
    Sigma_inv=Sigma_inv_v2,
    meta={'model': model_v2, 'spec': spec_v2, 'module': module_v2},
)
print(f'v2 target built in {time.perf_counter() - t0:.2f}s  D={target_v2.D}')

assert target_v1.D == target_v2.D

### Quick reference comparison

Adam with the same seed but different Python closures will land in slightly different places. Big differences here would be suspicious; small ones are expected.

In [ ]:
x_diff = (target_v1.x_ref - target_v2.x_ref).abs().max().item()
s_diff = (target_v1.Sigma_inv - target_v2.Sigma_inv).abs().max().item()
print(f'|x_ref_v1 - x_ref_v2|_inf      = {x_diff:.3e}')
print(f'|Sigma_inv_v1 - Sigma_inv_v2|  = {s_diff:.3e}')
print(f'||x_ref_v1||_inf = {target_v1.x_ref.abs().max().item():.3e}')
print(f'||x_ref_v2||_inf = {target_v2.x_ref.abs().max().item():.3e}')

## Run Boomerang on each

Same seed, same number of skeleton points, same refresh rate, same thinning. Only the `grad_target` closure differs.

In [ ]:
N_SKEL = 50_000
N_RESAMPLE = 5000
BURNIN_FRAC = 0.3
REFRESH_RATE = 0.1

def run_boomerang(target, seed=123):
    torch.manual_seed(seed); np.random.seed(seed)
    s = AutomaticBoomerangSampler(
        grad_target=target.grad_target, D=target.D,
        refresh_rate=REFRESH_RATE, thinning='pli',
    )
    s.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)
    x0 = target.x_ref.clone() + 0.1 * torch.randn(target.D, dtype=DTYPE)
    t0 = time.perf_counter()
    res = s.sample(N=N_SKEL, x0=x0, diagnostics=False)
    wall = time.perf_counter() - t0
    print(f'  {N_SKEL} skel in {wall:.2f}s, '
          f'final_t={res["times"][-1].item():.2f}, '
          f'grad_evals={res["gradient_evals"]}')
    return res

print('Running v1 ...')
res_v1 = run_boomerang(target_v1)
print('Running v2 ...')
res_v2 = run_boomerang(target_v2)

## Resample the continuous-time path to a discrete posterior chain

Both Boomerangs use the same anchor `x_ref` for the elliptical interpolation, so the resampler is called with each target's own `x_ref`.

In [ ]:
def resample_path(res, target):
    pos = res['positions'].cpu().numpy()
    vel = res['velocities'].cpu().numpy()
    tim = res['times'].cpu().numpy()
    x_ref_np = target.x_ref.cpu().numpy()
    samples_np = resample_boomerang_path(
        pos, vel, tim, x_ref_np,
        N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
    )
    return torch.tensor(samples_np, dtype=DTYPE)

samples_v1 = resample_path(res_v1, target_v1)
samples_v2 = resample_path(res_v2, target_v2)
print(f'samples_v1: {tuple(samples_v1.shape)}')
print(f'samples_v2: {tuple(samples_v2.shape)}')

## Predictive distributions on a dense grid

In [ ]:
# Dense grid covering the input range. The hernandez toy is on [-1, 1] roughly;
# adjust if needed for the dataset you're using.
x_min = float(min(X_train.min(), X_test.min())) - 0.2
x_max = float(max(X_train.max(), X_test.max())) + 0.2
x_grid = torch.linspace(x_min, x_max, 400, dtype=DTYPE).unsqueeze(-1)  # [400, 1]
print(f'grid: {tuple(x_grid.shape)}, range [{x_min:.2f}, {x_max:.2f}]')

@torch.no_grad()
def predict_v1(samples, X):
    """Forward pass through the v1 likelihood for each sample."""
    likelihood = target_v1.meta['model'].likelihood
    return torch.stack([likelihood.predict(b, X).squeeze(-1) for b in samples])

@torch.no_grad()
def predict_v2(samples, X):
    """Forward pass via functional_call for each sample."""
    out = []
    for b in samples:
        params = spec_v2.to_dict(b)
        out.append(torch.func.functional_call(module_v2, params, (X,)).squeeze(-1))
    return torch.stack(out)

# Use a subsample for speed (predicting through 2000 betas on 400 grid points is fine,
# but no need for all of them when computing the band)
N_PRED = 1000
idx_v1 = torch.randperm(samples_v1.shape[0])[:N_PRED]
idx_v2 = torch.randperm(samples_v2.shape[0])[:N_PRED]

preds_v1 = predict_v1(samples_v1[idx_v1], x_grid)
preds_v2 = predict_v2(samples_v2[idx_v2], x_grid)

mean_v1, std_v1 = preds_v1.mean(0), preds_v1.std(0)
mean_v2, std_v2 = preds_v2.mean(0), preds_v2.std(0)

print(f'pred mean range v1: [{mean_v1.min():.3f}, {mean_v1.max():.3f}]')
print(f'pred mean range v2: [{mean_v2.min():.3f}, {mean_v2.max():.3f}]')

## Plot 1: predictive bands overlaid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
x_np = x_grid.squeeze(-1).numpy()

for ax, mean, std, color, name in [
    (axes[0], mean_v1, std_v1, 'C0', 'v1 (layer_sizes + unflatten_params)'),
    (axes[1], mean_v2, std_v2, 'C1', 'v2 (ParamSpec + functional_call)'),
]:
    ax.fill_between(x_np, (mean - 2*std).numpy(), (mean + 2*std).numpy(),
                    color=color, alpha=0.25, label='±2σ band')
    ax.plot(x_np, mean.numpy(), color=color, lw=2, label='posterior mean')
    ax.scatter(X_train.squeeze(-1).numpy(), y_train.numpy(),
               s=18, c='black', alpha=0.6, label='train', zorder=3)
    ax.scatter(X_test.squeeze(-1).numpy(), y_test.numpy(),
               s=14, c='gray', alpha=0.5, marker='x', label='test', zorder=3)
    ax.set_title(name)
    ax.set_xlabel('x'); ax.legend(loc='best', fontsize=9)
axes[0].set_ylabel('y')
fig.suptitle(f'Predictive distributions — {DATASET}', fontsize=12)
plt.tight_layout()
plt.show()

## Plot 2: bands directly on top of each other

If the two pipelines target the same posterior, these should largely overlap.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

ax.fill_between(x_np, (mean_v1 - 2*std_v1).numpy(), (mean_v1 + 2*std_v1).numpy(),
                color='C0', alpha=0.20, label='v1 ±2σ')
ax.fill_between(x_np, (mean_v2 - 2*std_v2).numpy(), (mean_v2 + 2*std_v2).numpy(),
                color='C1', alpha=0.20, label='v2 ±2σ')
ax.plot(x_np, mean_v1.numpy(), color='C0', lw=2, label='v1 mean')
ax.plot(x_np, mean_v2.numpy(), color='C1', lw=2, ls='--', label='v2 mean')

ax.scatter(X_train.squeeze(-1).numpy(), y_train.numpy(),
           s=18, c='black', alpha=0.6, label='train', zorder=3)
ax.scatter(X_test.squeeze(-1).numpy(), y_test.numpy(),
           s=14, c='gray', alpha=0.5, marker='x', label='test', zorder=3)

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'v1 vs v2 predictive — {DATASET}')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## Plot 3: posterior sample functions

First moments matching is necessary but not sufficient. Drawing actual sample functions is a stronger check — if both pipelines produce similar *shapes* of curves with similar variability, they're sampling from the same distribution.

In [ ]:
N_SHOW = 30   # how many sample curves to draw
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax, preds, color, name in [
    (axes[0], preds_v1, 'C0', 'v1 sample curves'),
    (axes[1], preds_v2, 'C1', 'v2 sample curves'),
]:
    pick = torch.randperm(preds.shape[0])[:N_SHOW]
    for i in pick:
        ax.plot(x_np, preds[i].numpy(), color=color, lw=0.6, alpha=0.4)
    ax.scatter(X_train.squeeze(-1).numpy(), y_train.numpy(),
               s=18, c='black', alpha=0.7, zorder=3)
    ax.scatter(X_test.squeeze(-1).numpy(), y_test.numpy(),
               s=14, c='gray', alpha=0.5, marker='x', zorder=3)
    ax.set_title(f'{name} (N={N_SHOW})')
    ax.set_xlabel('x')
axes[0].set_ylabel('y')
fig.suptitle(f'Posterior sample functions — {DATASET}', fontsize=12)
plt.tight_layout()
plt.show()

## Plot 4: predictive marginals at a few specific x's

For a handful of x values, plot the posterior predictive density of f(x) under each pipeline. They should overlap well — these are 1D distributions that are easy to compare.

In [ ]:
# Pick a few representative x values
x_probe = torch.tensor([
    [(x_min + x_max) * 0.25],
    [(x_min + x_max) * 0.50],
    [(x_min + x_max) * 0.75],
], dtype=DTYPE)

preds_probe_v1 = predict_v1(samples_v1, x_probe)  # [N_resample, 3]
preds_probe_v2 = predict_v2(samples_v2, x_probe)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for k, ax in enumerate(axes):
    a = preds_probe_v1[:, k].numpy()
    b = preds_probe_v2[:, k].numpy()
    bins = np.linspace(
        min(a.min(), b.min()) - 0.05,
        max(a.max(), b.max()) + 0.05,
        50,
    )
    ax.hist(a, bins=bins, density=True, color='C0', alpha=0.5, label='v1')
    ax.hist(b, bins=bins, density=True, color='C1', alpha=0.5, label='v2')
    ax.set_title(f'p(f(x*) | data),  x* = {x_probe[k, 0].item():.2f}')
    ax.set_xlabel('f(x*)'); ax.legend()
plt.tight_layout()
plt.show()

## Numerical summary

In [ ]:
# Test-set RMSE and predictive log-lik using each pipeline's own samples
import math

@torch.no_grad()
def metrics(preds_test, y_test, noise_std):
    # preds_test: [N_samples, N_test]
    mean = preds_test.mean(0)
    std  = preds_test.std(0)
    rmse = ((mean - y_test) ** 2).mean().sqrt().item()
    total_var = std ** 2 + noise_std ** 2
    total_std = total_var.sqrt()
    ll = (-0.5 * ((y_test - mean) / total_std) ** 2
          - total_std.log()
          - 0.5 * math.log(2 * math.pi)).mean().item()
    return rmse, ll, std.mean().item()

preds_test_v1 = predict_v1(samples_v1, X_test)
preds_test_v2 = predict_v2(samples_v2, X_test)

rmse_v1, ll_v1, sm_v1 = metrics(preds_test_v1, y_test, cfg.noise_std)
rmse_v2, ll_v2, sm_v2 = metrics(preds_test_v2, y_test, cfg.noise_std)

print(f'{"":<6}  {"RMSE":<10}  {"log-lik":<10}  {"mean pred std":<14}')
print(f'{"v1":<6}  {rmse_v1:<10.4f}  {ll_v1:<10.4f}  {sm_v1:<14.4f}')
print(f'{"v2":<6}  {rmse_v2:<10.4f}  {ll_v2:<10.4f}  {sm_v2:<14.4f}')
print()
print(f'difference (v1 - v2):')
print(f'  RMSE: {rmse_v1 - rmse_v2:+.4f}')
print(f'  log-lik: {ll_v1 - ll_v2:+.4f}')
print(f'  mean pred std: {sm_v1 - sm_v2:+.4f}')

## Interpretation

If the two pipelines target the same posterior, you should see:

- The two predictive bands overlap visually almost everywhere.
- The dashed v2 mean curve sits inside the v1 band (and vice versa) along most of the input range.
- Sample function plots have similar character — same wiggliness, similar amplitude, similar uncertainty growth in regions far from data.
- Predictive marginal histograms overlap heavily.
- Test RMSE and log-lik agree to roughly two significant figures.

Where you might see legitimate differences:

- Tail behaviour far from the training data — the band can be slightly wider or narrower because both chains are only 5000 skeleton points and 1000 prediction samples each. Run more samples to reduce this.
- Subtle bias in `x_ref` between the two pipelines (Adam stochasticity) can shift the band by a small constant, especially in regions with no data.

If you see qualitatively different shapes (e.g. one band tracks a bend in the data and the other doesn't), that's a real signal that something is off — but I'd be very surprised given that the static check passed bit-for-bit on D, prior precision, kappa, energy, and gradient.

## Next steps

Once this confirms agreement on the FFN, the next thing to plot is the same predictive band but with a `Conv1D`-based v2 module. That tests the full pipeline on a non-FFN architecture.